# Modelo de Deteccion de Fraude en Tarjetas de Credito

**Objetivo:** Detectar si una transaccion es fraudulenta (Class=1) o no (Class=0).
La metrica principal es **PR AUC** porque hay muy pocas transacciones fraudulentas (0.17%).

**Dataset:** 284.807 transacciones con variables anonimizadas V1-V28 (resultado de PCA),
mas el monto (Amount) y el tiempo (Time).

Lo que hice:
- Use PR AUC en vez de ROC AUC como metrica porque con tanto desbalance el ROC puede ser enganoso
- Busque los mejores hiperparametros con GridSearch para RandomForest y AdaBoost
- Elegi el umbral con F-beta dandole mas peso al recall para no dejar pasar fraudes
- El modelo final es el que da mejor PR AUC en validacion

---
## 1. Importar librerias

In [ ]:
# ============================================================
# 1. IMPORTAR LIBRERIAS
# ============================================================
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (train_test_split, RepeatedStratifiedKFold,
                                     GridSearchCV)
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score, fbeta_score,
                             classification_report, confusion_matrix,
                             recall_score, precision_score)
import pickle

RANDOM_STATE = 42

---
## 2. Cargar datos

In [ ]:
# ============================================================
# 2. CARGAR DATOS
# ============================================================
df = pd.read_csv('creditcard.csv')

print(f'Shape: {df.shape}')
print(f'\nDistribucion de Class:')
print(df['Class'].value_counts())
print(f'\nProporcion de fraude: {df["Class"].mean()*100:.4f}%')
print(f'\nNulos: {df.isnull().sum().sum()}')

---
## 3. Analisis Exploratorio

El dataset ya viene con las variables V1-V28 transformadas por PCA, asi que no hay
que hacer mucho preprocesamiento. Lo que si hay que ajustar es Amount y Time porque
estan en una escala muy diferente a las demas variables.

In [ ]:
# ============================================================
# 3. ANALISIS EXPLORATORIO
# ============================================================
print('=== Estadisticas por clase ===')
print(df.groupby('Class')[['Amount', 'Time']].describe())

print('\n=== Estadisticas generales ===')
print(df.describe())

---
## 4. Preparacion de variables

Como las variables V1-V28 ya estan normalizadas por PCA, solo normalizo Amount y Time
con StandardScaler para que queden en la misma escala.

In [ ]:
# ============================================================
# 4. PREPARACION DE VARIABLES
# ============================================================
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled']   = scaler.fit_transform(df[['Time']])

# features finales: V1-V28 + Amount y Time normalizados
FEATURES = [f'V{i}' for i in range(1, 29)] + ['Amount_scaled', 'Time_scaled']

X = df[FEATURES]
y = df['Class']

print(f'Total features: {len(FEATURES)}')
print(f'Fraudes: {y.sum()} de {len(y)} transacciones')

---
## 5. Separar en train y validacion (80/20)

Uso stratify=y para que la proporcion de fraudes quede igual en las dos partes.

In [ ]:
# ============================================================
# 5. SPLIT TRAIN / VALIDACION
# ============================================================
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape} | Fraudes: {y_train.sum()}')
print(f'Val:   {X_val.shape}   | Fraudes: {y_val.sum()}')
print(f'Baseline PR AUC (prevalencia): {y_val.mean():.4f}')

---
## 6. GridSearch para RandomForest y AdaBoost

Busco los mejores hiperparametros para cada modelo usando validacion cruzada estratificada.
Uso PR AUC como metrica de scoring porque con tanto desbalance accuracy no sirve.

Esta celda tarda varios minutos.

In [ ]:
# ============================================================
# 6. GRIDSEARCH
# ============================================================
SCORING = 'average_precision'  # PR AUC

def grid_RandomForest(X_train, y_train):
    model = RandomForestClassifier(random_state=1, class_weight='balanced')
    param_grid = {
        'n_estimators'    : [200, 300],
        'max_depth'       : [8, 12],
        'min_samples_leaf': [5],
        'criterion'       : ['gini', 'entropy']
    }
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=1, random_state=1)
    gs = GridSearchCV(model, param_grid, cv=cv, scoring=SCORING, n_jobs=-1)
    gs.fit(X_train, y_train)
    print(f'Mejores parametros RF: {gs.best_params_}')
    print(f'Mejor PR AUC CV RF:    {gs.best_score_:.4f}')
    return gs.best_estimator_

def grid_Adaboost(X_train, y_train):
    model = AdaBoostClassifier(random_state=1)
    param_grid = {
        'n_estimators' : [60, 100, 200],
        'learning_rate': np.linspace(0.01, 1, 12)
    }
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=1, random_state=1)
    gs = GridSearchCV(model, param_grid, cv=cv, scoring=SCORING, n_jobs=-1)
    gs.fit(X_train, y_train)
    print(f'Mejores parametros ADA: {gs.best_params_}')
    print(f'Mejor PR AUC CV ADA:    {gs.best_score_:.4f}')
    return gs.best_estimator_


print('Buscando mejores parametros RandomForest...')
best_rf = grid_RandomForest(X_train, y_train)

print('\nBuscando mejores parametros AdaBoost...')
best_ada = grid_Adaboost(X_train, y_train)

---
## 7. Comparacion de modelos

Evaluo RandomForest, AdaBoost y un ensemble que promedia las probabilidades de los dos.
Me quedo con el que tenga mejor PR AUC en validacion.

In [ ]:
# ============================================================
# 7. COMPARACION DE MODELOS
# ============================================================
modelos = {'RandomForest': best_rf, 'AdaBoost': best_ada}
probas  = {}

print(f"{'Modelo':<14} {'ROC AUC':>9} {'PR AUC':>9}")
for nombre, m in modelos.items():
    p = m.predict_proba(X_val)[:, 1]
    probas[nombre] = p
    print(f"{nombre:<14} {roc_auc_score(y_val, p):9.4f} {average_precision_score(y_val, p):9.4f}")

# ensemble: promedio de probabilidades
proba_ens = np.mean(list(probas.values()), axis=0)
probas['ENSEMBLE'] = proba_ens
print(f"{'ENSEMBLE':<14} {roc_auc_score(y_val, proba_ens):9.4f} {average_precision_score(y_val, proba_ens):9.4f}")

# elijo el mejor por PR AUC
ap = {n: average_precision_score(y_val, p) for n, p in probas.items()}
mejor_nombre = max(ap, key=ap.get)
proba_val    = probas[mejor_nombre]
print(f'\n>>> Mejor modelo: {mejor_nombre} (PR AUC={ap[mejor_nombre]:.4f})')
print(f'    Baseline (azar): {y_val.mean():.4f} → x{ap[mejor_nombre]/y_val.mean():.1f} mejor')

---
## 8. Elegir el umbral con F-beta

F1 pesa recall y precision igual. Como no quiero dejar pasar fraudes, uso F-beta con
beta=2 para que el recall pese el doble que la precision.

In [ ]:
# ============================================================
# 8. ELECCION DEL UMBRAL CON F-BETA
# ============================================================
BETA = 2.0

grid_umbrales = np.round(np.arange(0.02, 0.90, 0.002), 4)
fbetas = [fbeta_score(y_val, (proba_val >= t).astype(int), beta=BETA, zero_division=0)
          for t in grid_umbrales]
umbral = float(grid_umbrales[int(np.argmax(fbetas))])

pred = (proba_val >= umbral).astype(int)
tn, fp, fn, tp = confusion_matrix(y_val, pred).ravel()

print(f'Umbral elegido (max F{BETA:g}): {umbral:.4f}')
print(f'Recall={recall_score(y_val, pred):.3f}  '
      f'Precision={precision_score(y_val, pred, zero_division=0):.3f}  '
      f'F{BETA:g}={fbeta_score(y_val, pred, beta=BETA, zero_division=0):.3f}')
print(f'Detecta {tp}/{tp+fn} fraudes | falsas alarmas: {fp}')
print('\n' + classification_report(y_val, pred, digits=4))

---
## 9. Comparar distintos valores de beta

Tabla para ver como cambia el recall segun cuanto peso le doy.

In [ ]:
# ============================================================
# 9. COMPARACION DE BETAS
# ============================================================
print(f"{'beta':>5} {'umbral':>8} {'recall':>8} {'prec':>7} {'F1':>7} {'flag%':>7}")
for b in [1.0, 2.0, 2.5, 3.0]:
    fb = [fbeta_score(y_val, (proba_val >= t).astype(int), beta=b, zero_division=0)
          for t in grid_umbrales]
    tb = grid_umbrales[int(np.argmax(fb))]
    pb = (proba_val >= tb).astype(int)
    print(f"{b:5.1f} {tb:8.3f} {recall_score(y_val, pb):8.3f}"
          f" {precision_score(y_val, pb, zero_division=0):7.3f}"
          f" {fbeta_score(y_val, pb, beta=1, zero_division=0):7.3f}"
          f" {100*pb.sum()/len(pb):7.1f}")

---
## 10. Entrenar modelo final y guardar para la app

Reentreno con el 100% de los datos y guardo el modelo con pickle para usarlo en la app Flask.

In [ ]:
# ============================================================
# 10. MODELO FINAL + GUARDAR
# ============================================================
if mejor_nombre == 'ENSEMBLE':
    for m in modelos.values():
        if hasattr(m, 'fit'):
            m.fit(X, y)
    modelo_final = modelos
    es_ensemble = True
else:
    modelo_final = modelos[mejor_nombre]
    modelo_final.fit(X, y)
    es_ensemble = False

# guardar modelo, scaler y umbral
with open('modelo_fraude.pkl', 'wb') as f:
    pickle.dump({
        'modelo'      : modelo_final,
        'scaler'      : scaler,
        'umbral'      : umbral,
        'features'    : FEATURES,
        'es_ensemble' : es_ensemble,
        'mejor_nombre': mejor_nombre
    }, f)

print('Modelo guardado en modelo_fraude.pkl')
print(f'Umbral final: {umbral}')
print(f'Modelo elegido: {mejor_nombre}')